## Handling missing data

**Sentinel:** placeholder for missing values (e.g. _NaN_)


In [2]:
import numpy as np
import pandas as pd

# get series excluding null data and indexes:
pd.Series([1, np.nan, 3, None, 5]).dropna()

0    1.0
2    3.0
4    5.0
dtype: float64

In [3]:
# in dataframes the whole row that contains at least one NA by default. we can specify 'how' param to change this:
data = pd.DataFrame(np.arange(12).reshape(4, 3))
data.iloc[1, 0] = np.nan
data.iloc[2] = None

data.dropna()

,0,1,2
0,0.0,1.0,2.0
3,9.0,10.0,11.0


In [4]:
data.dropna(how="all")

,0,1,2
0,0.0,1.0,2.0
1,NaN,4.0,5.0
3,9.0,10.0,11.0


In [5]:
# drop columns instead of rows:
data.iloc[2, -1] = 0
data.dropna(axis="columns")

,2
0,2.0
1,5.0
2,0.0
3,11.0


In [6]:
# param 'thresh' specifies a minimum number of non null values:
data.dropna(thresh=1)

,0,1,2
0,0.0,1.0,2.0
1,NaN,4.0,5.0
2,NaN,NaN,0.0
3,9.0,10.0,11.0


In [7]:
# we can fill with the same methods from reindex:
data.fillna(method="bfill")

C:\Users\biels\AppData\Local\Temp\ipykernel_9692\2143231105.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method="bfill")


,0,1,2
0,0.0,1.0,2.0
1,9.0,4.0,5.0
2,9.0,10.0,0.0
3,9.0,10.0,11.0


In [8]:
# fill with average:
data.fillna(data.mean())

,0,1,2
0,0.0,1.0,2.0
1,4.5,4.0,5.0
2,4.5,5.0,0.0
3,9.0,10.0,11.0


## Data transformation

### Removing duplicates


In [9]:
# duplicated returns a boolean series - True if row is a duplicate from an earlier row, false otherwise:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"], "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


In [10]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

In [11]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


In [12]:
data["v1"] = np.arange(7)

# use a subset of columns to evaluate whether it is a duplicate:
data.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1


In [13]:
# keep = 'last' will drop the first occurrences instead of the last ones:
data.drop_duplicates(subset=["k1"], keep="last")

,k1,k2,v1
4,one,3,4
6,two,4,6


### Transforming using function or mapping


In [14]:
data = pd.DataFrame(
    {
        "food": [
            "bacon",
            "pulled pork",
            "bacon",
            "pastrami",
            "corned beef",
            "bacon",
            "pastrami",
            "honey ham",
            "nova lox",
        ],
        "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6],
    }
)

meat_to_animal = {
    "bacon": "pig",
    "pulled pork": "pig",
    "pastrami": "cow",
    "corned beef": "cow",
    "honey ham": "pig",
    "nova lox": "salmon",
}

data["animal"] = data["food"].map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [15]:
# passing a function works too:
def get_animal(x):
    return meat_to_animal[x]


data["food"].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

### Replacing values


In [16]:
data = pd.Series([1, -999, 2, -999, -1000, 3])
data.replace(-999, np.nan)

0       1.0
1       NaN
2       2.0
3       NaN
4   -1000.0
5       3.0
dtype: float64

In [17]:
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

In [18]:
data.replace([-999, -1000], [np.nan, 0])
# data.replace({-999: np.nan, -1000: 0}) # does the same

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

### Renaming axis indexes


In [19]:
data = pd.DataFrame(
    np.arange(12).reshape((3, 4)), index=["Ohio", "Colorado", "New York"], columns=["one", "two", "three", "four"]
)


def transform(x):
    return x[:4].upper()


data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

In [20]:
data.index = data.index.map(transform)
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


In [21]:
data.rename(index=str.title, columns=str.upper)  # will return a transformed copy of the df

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


In [22]:
# dicts also work:
data.rename(index={"OHIO": "INDIANA"}, columns={"three": "peekaboo"})

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


### Discretization and Binning

Binning means separating continuous data into _bins_.


In [23]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

bins = [18, 25, 35, 60, 100]
age_categories = pd.cut(ages, bins)

age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

In [24]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [25]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [26]:
pd.value_counts(age_categories)

C:\Users\biels\AppData\Local\Temp\ipykernel_9692\3010498523.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(age_categories)


(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

In [27]:
# make sides open (exclusive) or closed (inclusive):
pd.cut(ages, bins, right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

In [28]:
# label categories:
labels = ["Youth", "YoungAdult", "MiddleAged", "Senior"]
pd.cut(ages, bins, labels=labels)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

In [29]:
# specify amount of bins instead of bin edges:
pd.cut(np.random.uniform(size=20), 4, precision=2)  # precision limits decimal precision

[(0.71, 0.95], (0.0078, 0.24], (0.48, 0.71], (0.24, 0.48], (0.24, 0.48], ..., (0.24, 0.48], (0.24, 0.48], (0.71, 0.95], (0.0078, 0.24], (0.24, 0.48]]
Length: 20
Categories (4, interval[float64, right]): [(0.0078, 0.24] < (0.24, 0.48] < (0.48, 0.71] < (0.71, 0.95]]

In [30]:
# cut based on quantiles - get "equally" sized bins:
quartiles = pd.qcut(np.random.standard_normal(1000), 4, precision=2)
quartiles

[(0.7, 3.26], (0.0041, 0.7], (-2.9299999999999997, -0.67], (-0.67, 0.0041], (0.7, 3.26], ..., (-2.9299999999999997, -0.67], (-2.9299999999999997, -0.67], (-0.67, 0.0041], (0.0041, 0.7], (-2.9299999999999997, -0.67]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.9299999999999997, -0.67] < (-0.67, 0.0041] < (0.0041, 0.7] < (0.7, 3.26]]

In [31]:
quartiles.value_counts()

(-2.9299999999999997, -0.67]    250
(-0.67, 0.0041]                 250
(0.0041, 0.7]                   250
(0.7, 3.26]                     250
Name: count, dtype: int64

In [32]:
# specify arbitrary quantiles:
pd.qcut(np.random.standard_normal(1000), [0, 0.1, 0.5, 1]).value_counts()

(-3.393, -1.234]    100
(-1.234, 0.0402]    400
(0.0402, 3.315]     500
Name: count, dtype: int64

### Detecting and filtering outliers


In [33]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.034749,-0.005678,-0.009326,0.056409
std,0.946529,1.026296,1.008298,0.970795
min,-3.215115,-2.964445,-3.260082,-3.061409
25%,-0.673881,-0.698559,-0.716546,-0.595115
50%,0.015882,-0.029390,0.029903,0.082729
75%,0.630196,0.603655,0.616442,0.698042
max,3.038796,3.032334,3.068561,3.568980


In [34]:
# select all rows where at least one column exceeds -3 .. 3:
data[(data.abs() > 3).any(axis="columns")]

,0,1,2,3
50,-1.207938,0.918481,1.472254,3.000771
55,3.038796,0.630524,0.784900,0.668644
102,-0.449829,0.869816,1.119825,-3.061409
131,-1.992828,-0.368314,3.068561,-1.639245
198,0.706766,3.032334,0.572538,0.273309
218,-0.264308,-2.242814,-3.107228,0.376759
387,0.643051,-0.177989,0.594079,3.568980
592,0.342331,0.143611,-3.260082,-0.804858
717,0.121137,0.163599,-3.162204,0.713926
875,-3.215115,-0.361545,-0.338331,-0.104575


In [35]:
# cap these values to +- 3:
data[data.abs() > 3] = np.sign(data) * 3
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.034573,-0.005711,-0.008865,0.055644
std,0.945705,1.026201,1.006464,0.967881
min,-3.000000,-2.964445,-3.000000,-3.000000
25%,-0.673881,-0.698559,-0.716546,-0.595115
50%,0.015882,-0.029390,0.029903,0.082729
75%,0.630196,0.603655,0.616442,0.698042
max,3.000000,3.000000,3.000000,3.000000


### Permutation and random sampling


In [ ]:
df = pd.DataFrame(np.arange(35).reshape((5, 7)))
sampler = np.random.permutation(5)  # produces an array of integers indicating the new ordering
sampler

array([4, 2, 0, 1, 3], dtype=int32)

In [ ]:
df.take(sampler)  # df.iloc[sampler] works too

,0,1,2,3,4,5,6
4,28,29,30,31,32,33,34
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
3,21,22,23,24,25,26,27


In [ ]:
# same thing for columns:
column_sampler = np.random.permutation(7)
column_sampler

array([1, 0, 6, 2, 5, 4, 3], dtype=int32)

In [ ]:
# df.take(column_sampler, axis='columns')
df.iloc[:, column_sampler]

,1,0,6,2,5,4,3
0,1,0,6,2,5,4,3
1,8,7,13,9,12,11,10
2,15,14,20,16,19,18,17
3,22,21,27,23,26,25,24
4,29,28,34,30,33,32,31


In [ ]:
df.sample(n=3)  # select 3 random rows, no repetition. replace=True allows repetition

,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27


### Computing indicator/Dummy variables

Converting a category represented as number to many columns where, for each row, one is `True` (or `1`) and the others are `False` (or `0`)

Useful for _one-hot encoding_ in machine learning modelling.


In [ ]:
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"], "data1": range(6)})
df

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [ ]:
dummies = pd.get_dummies(
    df["key"],  # column
    dtype=float,  # default is boolean
    prefix="key",  # prefix for new column names
)
dummies

,key_a,key_b,key_c
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
5,0.0,1.0,0.0


In [ ]:
# joining with original df:
df[["data1"]].join(dummies)

,data1,key_a,key_b,key_c
0,0,0.0,1.0,0.0
1,1,0.0,1.0,0.0
2,2,1.0,0.0,0.0
3,3,0.0,0.0,1.0
4,4,1.0,0.0,0.0
5,5,0.0,1.0,0.0


In [ ]:
# approach for when one row belongs to multiple categories:
mnames = ["movie_id", "title", "genres"]
movies = pd.read_table("datasets/movielens/movies.dat", sep="::", header=None, names=mnames, engine="python")
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
dummies = movies["genres"].str.get_dummies("|")
dummies.iloc[:19, :6]

,Action,Adventure,Animation,Children's,Comedy,Crime
0,0,0,1,1,1,0
1,0,1,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,1,0
4,0,0,0,0,1,0
5,1,0,0,0,0,1
6,0,0,0,0,1,0
7,0,1,0,1,0,0
8,1,0,0,0,0,0
9,1,1,0,0,0,0


In [ ]:
movies_with_dummies = movies.join(dummies.add_prefix("genre_"))
movies_with_dummies.iloc[0]

movie_id                                       1
title                           Toy Story (1995)
genres               Animation|Children's|Comedy
genre_Action                                   0
genre_Adventure                                0
genre_Animation                                1
genre_Children's                               1
genre_Comedy                                   1
genre_Crime                                    0
genre_Documentary                              0
genre_Drama                                    0
genre_Fantasy                                  0
genre_Film-Noir                                0
genre_Horror                                   0
genre_Musical                                  0
genre_Mystery                                  0
genre_Romance                                  0
genre_Sci-Fi                                   0
genre_Thriller                                 0
genre_War                                      0
genre_Western       

In [56]:
# combining get_dummies with cut:
np.random.seed(12345)
values = np.random.uniform(size=10)
values

array([0.92961609, 0.31637555, 0.18391881, 0.20456028, 0.56772503,
       0.5955447 , 0.96451452, 0.6531771 , 0.74890664, 0.65356987])

In [ ]:
bins = np.linspace(0, 1, 6)  # 5 equally spaced bins
bins

array([0. , 0.2, 0.4, 0.6, 0.8, 1. ])

In [60]:
pd.get_dummies(pd.cut(values, bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False


## Extension data types


In [ ]:
s = pd.Series([1, 2, 3, None])  # dtype is float64
s

0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64

In [63]:
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())
s

0       1
1       2
2       3
3    <NA>
dtype: Int64

In [64]:
s.isna()

0    False
1    False
2    False
3     True
dtype: bool

In [ ]:
# for strings - computationally more optimal:
pd.Series(["one", "two", None, "three"], dtype=pd.StringDtype())

0      one
1      two
2     <NA>
3    three
dtype: string

In [ ]:
# we can convert types on the fly too:
df = pd.DataFrame({"A": [1, 2, None, 4], "B": ["one", "two", "three", None], "C": [False, None, False, True]})
df

,A,B,C
0,1.0,one,False
1,2.0,two,None
2,NaN,three,False
3,4.0,None,True


In [67]:
df["A"] = df["A"].astype("Int64")  # sane as pd.Int64Dtype()
df["B"] = df["B"].astype("string")
df["C"] = df["C"].astype("boolean")

df

,A,B,C
0,1,one,False
1,2,two,<NA>
2,<NA>,three,False
3,4,<NA>,True


## String manipulation

### String functions in pandas


In [ ]:
data = pd.Series({"Dave": "dave@google.com", "Steve": "steve@gmail.com", "Rob": "rob@gmail.com", "Wes": np.nan})
data.str.contains("gmail")  # will skip null values for us

Dave     False
Steve     True
Rob       True
Wes        NaN
dtype: object

In [ ]:
# using regular expressions:
import re

pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]
matches

Dave     (dave, google, com)
Steve    (steve, gmail, com)
Rob        (rob, gmail, com)
Wes                      NaN
dtype: object

In [ ]:
matches.str.get(1)  # get only second string in each tuple

Dave     google
Steve     gmail
Rob       gmail
Wes         NaN
dtype: object

In [76]:
data.str[5:]

Dave     google.com
Steve    @gmail.com
Rob        mail.com
Wes             NaN
dtype: object

In [77]:
# convert regex groups to separate columns
data.str.extract(pattern, flags=re.IGNORECASE)

,0,1,2
Dave,dave,google,com
Steve,steve,gmail,com
Rob,rob,gmail,com
Wes,NaN,NaN,NaN


## Categorical data


In [ ]:
# common strategy for representing categorical columns: "dictionary=encoded" - store integers and have a reference table to lookup actual values against
# instead of representing a column that the value is either apples or oranges like this:
values = pd.Series(["apple", "orange", "apple", "apple"] * 2)
values

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
dtype: object

In [ ]:
# represent it like this - will repeat integers instead of whole strings:
values = pd.Series([0, 1, 0, 0] * 2)
dim = pd.Series(["apple", "orange"])
dim.take(values)

0     apple
1    orange
0     apple
0     apple
0     apple
1    orange
0     apple
0     apple
dtype: object

### `Categorical` extension type


In [80]:
fruits = ["apple", "orange", "apple", "apple"] * 2
N = len(fruits)
rng = np.random.default_rng(seed=12345)
df = pd.DataFrame(
    {
        "fruit": fruits,
        "basket_id": np.arange(N),
        "count": rng.integers(3, 15, size=N),
        "weight": rng.uniform(0, 4, size=N),
    },
    columns=["basket_id", "fruit", "count", "weight"],
)
df

,basket_id,fruit,count,weight
0,0,apple,11,1.564438
1,1,orange,5,1.331256
2,2,apple,12,2.393235
3,3,apple,6,0.746937
4,4,apple,5,2.691024
5,5,orange,12,3.767211
6,6,apple,10,0.992983
7,7,apple,11,3.795525


In [ ]:
fruit_cat = df["fruit"].astype("category")
fruit_cat

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

In [82]:
fruit_cat.array

['apple', 'orange', 'apple', 'apple', 'apple', 'orange', 'apple', 'apple']
Categories (2, object): ['apple', 'orange']

In [83]:
fruit_cat.array.categories

Index(['apple', 'orange'], dtype='object')

In [84]:
fruit_cat.array.codes

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

In [85]:
dict(enumerate(fruit_cat.array.categories))

{0: 'apple', 1: 'orange'}

In [ ]:
df["fruit"] = df["fruit"].astype("category")

In [ ]:
# create directly from python sequences:
pd.Categorical(["foo", "bar", "baz", "foo", "bar"])

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, object): ['bar', 'baz', 'foo']

In [ ]:
categories = ["foo", "bar", "baz"]
codes = [0, 1, 2, 0, 0, 1]
pd.Categorical.from_codes(codes, categories)

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo', 'bar', 'baz']

In [91]:
# specifying order between categories:
pd.Categorical.from_codes(codes, categories, ordered=True)

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

In [ ]:
# verifying the difference in memory:
N = 10_000_000
labels = pd.Series(["foo", "bar", "baz", "qux"] * (N // 4))
labels.memory_usage()

80000128

In [ ]:
categories = labels.astype("category")
categories.memory_usage()

10000332

In [94]:
%timeit labels.value_counts()

361 ms ± 38.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [95]:
%timeit categories.value_counts()

45.9 ms ± 3.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
# special attribute `cat`
cat_s = pd.Series(list("abcd") * 2, dtype="category")
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [106]:
cat_s.cat.codes

0    0
1    1
2    2
3    3
4    0
5    1
6    2
7    3
dtype: int8

In [101]:
cat_s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='object')

In [103]:
# create a new category that doesn't exist in the rows:
actual_cats = list("abcde")
cat_s2 = cat_s.cat.set_categories(actual_cats)  # returns a new Series
cat_s2

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (5, object): ['a', 'b', 'c', 'd', 'e']

In [104]:
# will affect other operations even though there are no elements of category 'e':
cat_s2.value_counts()

a    2
b    2
c    2
d    2
e    0
Name: count, dtype: int64

In [105]:
# remove unused categories:
cat_s2.cat.remove_unused_categories()

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']